# Day 6 — CNN Product Image Classification

Fashion-MNIST validates the architecture. The public Kaggle Fashion Product Images dataset is then downloaded through `tf.keras.utils.get_file`, cached outside this repository, and used to train RetailMind AI's first catalog classifier.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
from src.catalog_classifier import (
    FASHION_MNIST_LABELS, build_catalog_cnn, download_fashion_product_images,
    fashion_mnist_data, fashion_product_datasets, save_catalog_classifier,
    save_catalog_metrics,
)

tf.keras.utils.set_random_seed(42)
print('TensorFlow:', tf.__version__)

## Fashion-MNIST prototype

The model follows conv → pool → dropout → dense. Its test split remains untouched until evaluation.

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist_data()
fashion_model = build_catalog_cnn((28, 28, 1), len(FASHION_MNIST_LABELS))
fashion_model.fit(x_train, y_train, validation_split=0.1, epochs=5, batch_size=128, verbose=2)
_, fashion_accuracy = fashion_model.evaluate(x_test, y_test, verbose=0)
fashion_predictions = tf.argmax(fashion_model.predict(x_test, verbose=0), axis=1, output_type=tf.int32)
fashion_matrix = tf.math.confusion_matrix(y_test, fashion_predictions, num_classes=len(FASHION_MNIST_LABELS))
print(f'Fashion-MNIST test accuracy: {fashion_accuracy:.4%}')
print('Fashion-MNIST confusion matrix (actual rows, predicted columns):')
print(fashion_matrix.numpy())

## Product catalog classifier

The next cell imports `paramaggarwal/fashion-product-images-small` automatically. It reads `styles.csv` and image files from Keras' cache, classifies `subCategory`, uses a deterministic per-category 80:20 split, and augments only training images.

In [ ]:
dataset_dir = download_fashion_product_images()
print('Kaggle dataset cache:', dataset_dir)

train_ds, validation_ds, class_names, dataset_summary = fashion_product_datasets(
    dataset_dir, label_column='subCategory', image_size=(128, 128), batch_size=32,
    min_examples_per_class=100, seed=42,
)
print(dataset_summary)
print('Categories:', class_names)

catalog_model = build_catalog_cnn((128, 128, 3), len(class_names))
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', mode='max', patience=3, restore_best_weights=True
)
catalog_model.fit(train_ds, validation_data=validation_ds, epochs=15, callbacks=[early_stopping], verbose=2)
_, catalog_accuracy = catalog_model.evaluate(validation_ds, verbose=0)
catalog_probabilities = catalog_model.predict(validation_ds, verbose=0)
catalog_truth = tf.concat([labels for _, labels in validation_ds], axis=0)
catalog_predictions = tf.argmax(catalog_probabilities, axis=1, output_type=tf.int32)
catalog_matrix = tf.math.confusion_matrix(catalog_truth, catalog_predictions, num_classes=len(class_names))
model_path, labels_path = save_catalog_classifier(
    catalog_model, class_names, PROJECT_ROOT / 'models' / 'catalog_cnn_day6.keras'
)
metrics_path = save_catalog_metrics(model_path, catalog_accuracy, dataset_summary)

readme_path = PROJECT_ROOT / 'README.md'
readme = readme_path.read_text(encoding='utf-8')
old_line = '| Day 6 catalog CNN (Fashion Product Images validation split) | Not recorded - run Day 6 |'
new_line = f'| Day 6 catalog CNN (Fashion Product Images validation split) | {catalog_accuracy:.2%} |'
if old_line in readme:
    readme_path.write_text(readme.replace(old_line, new_line), encoding='utf-8')

print(f'Catalog validation accuracy: {catalog_accuracy:.4%}')
print('Catalog confusion matrix (actual rows, predicted columns):')
print(catalog_matrix.numpy())
print('Saved model:', model_path)
print('Saved category mapping:', labels_path)
print('Saved metrics:', metrics_path)
print('README accuracy row:', new_line)